In [15]:
import argparse
import sys
import re
from pathlib import Path
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Union
from openpyxl import load_workbook



In [26]:
def drop_bad_reads(df, times=[], images=[]):
    df = df.copy()
    
    if times and "Elapsed" in df.columns:
    # keep rows where Elapsed is NOT in the times list using Series.isin
        df = df[~df["Elapsed"].isin(times)]    
    if images:
        cols = df.columns.astype(str)
        cols_to_drop = []
        for col in cols:
            if ", Image" in col:
                m = re.search(r'Image\s*(\d+)$', col)
                if m:
                    try:
                        num = int(m.group(1))
                        if num in images:
                            cols_to_drop.append(col)
                    except ValueError:
                        pass
        if cols_to_drop:
            df = df.drop(columns=cols_to_drop)
           
    return df.reset_index(drop=True)

def make_average_column_for_well(df, num_reads_per_well = 9):
    df = df.copy()
    new_averages = {}
    cols = df.columns.astype(str)
    #only use numeric cols
    numeric_cols = cols[3:]
    #print(f"len {len(numeric_cols)}, reads used: {num_reads_per_well}, remainder: {len(numeric_cols) % num_reads_per_well}")
    
    if len(numeric_cols) % num_reads_per_well == 0: #make sure the number won't go out of range
        for i in range(0, len(numeric_cols), num_reads_per_well):
            cols_to_avg = [numeric_cols[i] for i in range(i,i+num_reads_per_well)]
            split_header = cols_to_avg[0].split(",")
            well_name = split_header[0]
            new_averages[well_name] = df[cols_to_avg].mean(axis=1)
        header_df = df[cols[:3]]
        
        new_df = pd.DataFrame(new_averages)
        return pd.concat([header_df,new_df], axis=1)
    else:
        return pd.DataFrame()

def make_technical_rep_average_columns(df, num_to_avg=2):
    df = df.copy()
    new_averages = {}
    cols = df.columns.astype(str)
    numeric_cols = cols[3:]
    
    if len(numeric_cols) % num_to_avg == 0: #make sure the number won't go out of range
        for i in range(0, len(numeric_cols), num_to_avg):
            cols_to_avg = [numeric_cols[i] for i in range(i,i+num_to_avg)]
            split_header = cols_to_avg[0].split("(")
            well_name = split_header[0]
            new_averages[well_name] = df[cols_to_avg].mean(axis=1)
        header_df = df[cols[:3]]
        
        new_avg_df = pd.DataFrame(new_averages)
        return pd.concat([header_df,new_avg_df], axis=1)
    else:
        return pd.DataFrame()


In [29]:
def make_average_excel_with_exclusions(
input,
output,
inplace=False,
input_sheet = "Reformatted",
output_sheet="Averages",
overwrite=False,
time_rows_to_drop = [0,2],
images_to_drop = [5],
display_table=False
):
    inp = Path(input)
    if not inp.exists():
        print(f"Input file does not exist: {inp}", file=sys.stderr)
        sys.exit(2)

    if inplace and output:
        print("Cannot use --inplace and --output together.", file=sys.stderr)
        sys.exit(2)

    if input_sheet is not None:
        # allow numeric index if user provides digits
        try:
            sheet_name = int(input_sheet)
        except ValueError:
            sheet_name = input_sheet
    else:
        sheet_name = "Sheet1"

    df = pd.read_excel(inp, sheet_name=sheet_name)
    
    # apply replacements
    drop_df = drop_bad_reads(df,time_rows_to_drop,images_to_drop)
    
    number_images_dropped = len(images_to_drop)
    increment = int(9-number_images_dropped)
    out_df = make_average_column_for_well(drop_df, increment)
    # avg the two wells per lineage/group
    out_df_avg = make_technical_rep_average_columns(out_df)
    if display_table:
        display(df)
        display(drop_df)
        display(out_df)
        display(out_df_avg)

    # determine output path
    if inplace:
        out_path = inp
    elif output:
        out_path = Path(output)
    else:
        out_path = inp.with_name(inp.stem + "_editded" + inp.suffix)

    if Path.exists(out_path):
        try:
            with pd.ExcelWriter(out_path, engine="openpyxl", mode="a") as writer:
                out_df.to_excel(writer, sheet_name=output_sheet)
                out_df_avg.to_excel(writer, sheet_name="Mean_"+output_sheet)
        except ValueError:
            if overwrite:
                with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as writer:
                    out_df.to_excel(writer, sheet_name=output_sheet)
                    out_df_avg.to_excel(writer, sheet_name="Mean_"+output_sheet)
            else:
                ValueError("Overwrite is disabled and sheet exists")
    else:
        # write back to excel
        out_df.to_excel(out_path, sheet_name=output_sheet, index=False)
    print(f"Wrote replaced data to: {out_path}")






In [ ]:
for i in range(1,8):
    plate = i
    input= f"/Users/allielas/HTP_fibroblasts_mitolyso/proliferation_growth_curves/Incucyte_sheets/r{plate}.xlsx"
    output= f"/Users/allielas/HTP_fibroblasts_mitolyso/proliferation_growth_curves/Incucyte_sheets/processed/r{plate}_processed.xlsx"
    make_average_excel_with_exclusions(input,output,display_table=True, overwrite=True)


/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


,Plate_Number,Vessel_Name,Elapsed,"P20 LIN1-0B-b1 (A1), Image 1","P20 LIN1-0B-b1 (A1), Image 2","P20 LIN1-0B-b1 (A1), Image 3","P20 LIN1-0B-b1 (A1), Image 4","P20 LIN1-0B-b1 (A1), Image 5","P20 LIN1-0B-b1 (A1), Image 6","P20 LIN1-0B-b1 (A1), Image 7",...,"P10 LIN5-0B-b3 (C3), Image 9","P10 LIN5-0B-b3 (C4), Image 1","P10 LIN5-0B-b3 (C4), Image 2","P10 LIN5-0B-b3 (C4), Image 3","P10 LIN5-0B-b3 (C4), Image 4","P10 LIN5-0B-b3 (C4), Image 5","P10 LIN5-0B-b3 (C4), Image 6","P10 LIN5-0B-b3 (C4), Image 7","P10 LIN5-0B-b3 (C4), Image 8","P10 LIN5-0B-b3 (C4), Image 9"
0,1.0,AS_MRC-5_20240313,0,1.590103e+01,19.931510,20.285450,20.816760,47.851630,21.158760,26.945550,...,17.59704,17.47712,24.42150,33.82525,26.85103,35.82420,23.74823,26.88886,25.71194,20.37663
1,1.0,AS_MRC-5_20240313,4,1.370022e+01,17.716070,17.296350,18.042100,49.066320,21.104470,25.140340,...,15.45249,14.79670,23.98847,32.30291,24.50298,36.44600,25.45489,23.20039,21.77953,18.32537
2,1.0,AS_MRC-5_20240313,8,1.463772e+01,18.375150,17.046340,19.791510,48.511180,20.220990,26.844000,...,18.82457,16.54188,25.78643,31.80896,26.68330,39.44445,29.67746,28.86678,23.64510,22.01076
3,1.0,AS_MRC-5_20240313,12,1.313210e+01,18.823890,19.819710,19.826740,52.072350,22.608780,27.270460,...,21.14353,18.09188,28.42391,37.00899,29.36483,47.95085,32.72959,30.47305,28.26028,23.69120
4,1.0,AS_MRC-5_20240313,16,1.318421e+01,20.718420,18.322290,20.089190,54.867390,26.392390,28.844040,...,22.26146,18.22026,29.39931,40.33483,33.08198,49.10907,36.21591,30.77803,33.11140,24.87578
5,1.0,AS_MRC-5_20240313,20,1.547025e+01,22.952910,21.085960,23.228260,61.578620,29.727380,32.613770,...,24.01517,22.72577,34.75934,46.19578,36.71526,63.43374,48.07835,38.70916,38.56288,31.00169
6,1.0,AS_MRC-5_20240313,24,1.639737e+01,25.823860,25.148330,25.560530,67.732600,35.175650,34.817380,...,29.80222,27.30830,40.11378,55.10899,44.07670,69.36155,53.26055,43.06381,42.44953,36.12196
7,1.0,AS_MRC-5_20240313,28,1.962740e+01,27.449330,28.997550,28.801150,70.922130,41.234420,41.416970,...,37.20976,30.22542,46.41178,62.42372,53.79835,76.57376,62.98090,50.15310,52.16469,42.12447
8,1.0,AS_MRC-5_20240313,32,2.261706e+01,30.780090,35.398410,33.435170,79.347760,47.933570,51.082280,...,45.47373,38.78162,57.61432,74.12410,66.96979,89.05362,72.18811,60.29761,62.47507,51.56707
9,1.0,AS_MRC-5_20240313,36,2.671288e+01,34.064960,38.748420,37.972370,81.647250,52.899430,57.348590,...,56.28858,44.38347,67.17063,81.80172,76.67517,95.69930,85.92998,70.13753,71.94984,62.81667


,Plate_Number,Vessel_Name,Elapsed,"P20 LIN1-0B-b1 (A1), Image 1","P20 LIN1-0B-b1 (A1), Image 2","P20 LIN1-0B-b1 (A1), Image 3","P20 LIN1-0B-b1 (A1), Image 4","P20 LIN1-0B-b1 (A1), Image 6","P20 LIN1-0B-b1 (A1), Image 7","P20 LIN1-0B-b1 (A1), Image 8",...,"P10 LIN5-0B-b3 (C3), Image 8","P10 LIN5-0B-b3 (C3), Image 9","P10 LIN5-0B-b3 (C4), Image 1","P10 LIN5-0B-b3 (C4), Image 2","P10 LIN5-0B-b3 (C4), Image 3","P10 LIN5-0B-b3 (C4), Image 4","P10 LIN5-0B-b3 (C4), Image 6","P10 LIN5-0B-b3 (C4), Image 7","P10 LIN5-0B-b3 (C4), Image 8","P10 LIN5-0B-b3 (C4), Image 9"
0,1.0,AS_MRC-5_20240313,4,1.370022e+01,17.716070,17.296350,18.042100,21.104470,25.140340,30.419580,...,19.36441,15.45249,14.79670,23.98847,32.30291,24.50298,25.45489,23.20039,21.77953,18.32537
1,1.0,AS_MRC-5_20240313,8,1.463772e+01,18.375150,17.046340,19.791510,20.220990,26.844000,31.502880,...,22.44311,18.82457,16.54188,25.78643,31.80896,26.68330,29.67746,28.86678,23.64510,22.01076
2,1.0,AS_MRC-5_20240313,12,1.313210e+01,18.823890,19.819710,19.826740,22.608780,27.270460,34.557330,...,24.51787,21.14353,18.09188,28.42391,37.00899,29.36483,32.72959,30.47305,28.26028,23.69120
3,1.0,AS_MRC-5_20240313,16,1.318421e+01,20.718420,18.322290,20.089190,26.392390,28.844040,35.515190,...,29.55665,22.26146,18.22026,29.39931,40.33483,33.08198,36.21591,30.77803,33.11140,24.87578
4,1.0,AS_MRC-5_20240313,20,1.547025e+01,22.952910,21.085960,23.228260,29.727380,32.613770,40.579310,...,36.07504,24.01517,22.72577,34.75934,46.19578,36.71526,48.07835,38.70916,38.56288,31.00169
5,1.0,AS_MRC-5_20240313,24,1.639737e+01,25.823860,25.148330,25.560530,35.175650,34.817380,44.017220,...,40.30909,29.80222,27.30830,40.11378,55.10899,44.07670,53.26055,43.06381,42.44953,36.12196
6,1.0,AS_MRC-5_20240313,28,1.962740e+01,27.449330,28.997550,28.801150,41.234420,41.416970,51.995800,...,50.45570,37.20976,30.22542,46.41178,62.42372,53.79835,62.98090,50.15310,52.16469,42.12447
7,1.0,AS_MRC-5_20240313,32,2.261706e+01,30.780090,35.398410,33.435170,47.933570,51.082280,59.191030,...,62.12774,45.47373,38.78162,57.61432,74.12410,66.96979,72.18811,60.29761,62.47507,51.56707
8,1.0,AS_MRC-5_20240313,36,2.671288e+01,34.064960,38.748420,37.972370,52.899430,57.348590,66.310570,...,69.15783,56.28858,44.38347,67.17063,81.80172,76.67517,85.92998,70.13753,71.94984,62.81667
9,1.0,AS_MRC-5_20240313,40,2.963847e+01,37.437440,42.312270,40.760070,59.686130,63.014300,74.004520,...,79.96462,66.73042,52.52240,72.74024,87.57976,83.77704,91.83469,75.79231,80.37989,73.81337


,Plate_Number,Vessel_Name,Elapsed,P20 LIN1-0B-b1 (A1),P20 LIN1-0B-b1 (B1),P17 LIN2-0A-b1 (A2),P17 LIN2-0A-b1 (B2),P14 LIN3-0A-b2 (A3),P14 LIN3-0A-b2 (B3),P12 LIN4-0B-b2 (A4),P12 LIN4-0B-b2 (B4),C1,C2,P10 LIN5-0B-b3 (C3),P10 LIN5-0B-b3 (C4)
0,1.0,AS_MRC-5_20240313,4,20.021759,18.021316,32.152679,12.916889,24.541841,21.079869,31.901086,13.265894,17.032940,31.826758,23.107985,23.043905
1,1.0,AS_MRC-5_20240313,8,20.756479,18.137957,33.343212,13.346433,27.592312,23.780141,33.973711,14.791917,17.545873,33.877419,26.193512,25.627584
2,1.0,AS_MRC-5_20240313,12,21.789315,19.331816,36.524350,15.537637,30.519455,25.993291,38.555700,17.217079,18.373919,37.354992,29.424637,28.505466
3,1.0,AS_MRC-5_20240313,16,22.848472,19.108665,39.675745,16.425696,33.762156,29.882112,45.497210,19.649744,18.930658,39.147137,32.338301,30.752187
4,1.0,AS_MRC-5_20240313,20,26.154750,20.556685,45.249935,16.947764,39.445140,34.300853,51.767805,22.553123,20.447750,42.600436,37.975759,37.093529
5,1.0,AS_MRC-5_20240313,24,29.246491,24.936691,52.857083,20.113686,45.959105,40.732365,61.928095,26.423706,23.120259,50.172835,44.369518,42.687952
6,1.0,AS_MRC-5_20240313,28,33.809222,27.444049,60.777354,23.747234,54.751840,47.699571,73.547412,31.269147,26.146615,56.825667,52.091890,50.035304
7,1.0,AS_MRC-5_20240313,32,39.736738,33.445327,71.599707,28.412880,65.659705,58.334918,84.467848,39.207499,30.513419,67.968046,63.236524,60.502211
8,1.0,AS_MRC-5_20240313,36,44.803481,37.575611,79.109641,32.345991,74.719066,67.555896,92.100022,47.515235,35.786423,73.734844,72.091574,70.108126
9,1.0,AS_MRC-5_20240313,40,49.592753,40.155701,85.044359,35.844299,81.312952,74.686950,95.162323,54.903931,38.123386,76.472599,80.979512,77.304963


,Plate_Number,Vessel_Name,Elapsed,P20 LIN1-0B-b1,P17 LIN2-0A-b1,P14 LIN3-0A-b2,P12 LIN4-0B-b2,C1,P10 LIN5-0B-b3
0,1.0,AS_MRC-5_20240313,4,19.021538,22.534784,22.810855,22.583490,24.429849,23.075945
1,1.0,AS_MRC-5_20240313,8,19.447218,23.344823,25.686227,24.382814,25.711646,25.910548
2,1.0,AS_MRC-5_20240313,12,20.560566,26.030993,28.256373,27.886389,27.864456,28.965052
3,1.0,AS_MRC-5_20240313,16,20.978569,28.050721,31.822134,32.573477,29.038898,31.545244
4,1.0,AS_MRC-5_20240313,20,23.355718,31.098849,36.872996,37.160464,31.524093,37.534644
5,1.0,AS_MRC-5_20240313,24,27.091591,36.485384,43.345735,44.175901,36.646547,43.528735
6,1.0,AS_MRC-5_20240313,28,30.626636,42.262294,51.225706,52.408280,41.486141,51.063597
7,1.0,AS_MRC-5_20240313,32,36.591032,50.006294,61.997311,61.837673,49.240733,61.869367
8,1.0,AS_MRC-5_20240313,36,41.189546,55.727816,71.137481,69.807629,54.760633,71.099850
9,1.0,AS_MRC-5_20240313,40,44.874227,60.444329,77.999951,75.033127,57.297992,79.142237


...

,Plate_Number,Vessel_Name,Elapsed,"Doxo_P19 LIN12-0C-b2-s2 (A1), Image 1","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 2","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 3","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 4","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 5","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 6","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 7",...,"Blank (C3), Image 9","Blank (C4), Image 1","Blank (C4), Image 2","Blank (C4), Image 3","Blank (C4), Image 4","Blank (C4), Image 5","Blank (C4), Image 6","Blank (C4), Image 7","Blank (C4), Image 8","Blank (C4), Image 9"
0,7,20250501_AS_MRC-5_Confluence,0,14.37534,15.92760,1.806777,3.849841,87.15396,5.606152,5.800508,...,0.380996,0.293447,0.033872,0.123743,0.141499,0.638590,0.122719,0.212795,0.625819,0.286071
1,7,20250501_AS_MRC-5_Confluence,2,15.88587,15.96686,3.029051,3.373511,88.91260,6.081868,5.244959,...,0.234716,0.311544,0.087754,0.257799,0.191420,1.701336,0.483159,0.162396,0.616668,0.756870
2,7,20250501_AS_MRC-5_Confluence,4,15.77353,16.47550,3.036290,3.456963,90.30471,7.190301,6.018014,...,0.525090,0.445873,0.272550,0.336607,0.672531,5.245575,0.466838,0.182610,5.572279,0.658258
3,7,20250501_AS_MRC-5_Confluence,6,16.08801,16.31747,3.001666,3.587808,89.99898,5.811367,5.851794,...,0.471140,0.527549,0.139928,0.213478,0.710091,7.566994,0.336334,0.214161,1.539691,0.213068
4,7,20250501_AS_MRC-5_Confluence,8,16.07449,16.58627,2.832714,3.704177,88.74133,5.470730,6.220634,...,0.208493,0.334695,0.227819,0.113500,0.139860,10.554120,0.396703,0.083042,2.310151,0.615439
5,7,20250501_AS_MRC-5_Confluence,10,17.27983,16.71561,3.338751,3.628578,88.84970,5.389533,5.053198,...,0.295769,0.279925,0.027180,0.029092,0.363513,11.368350,0.357231,0.114183,1.886336,0.309085
6,7,20250501_AS_MRC-5_Confluence,12,16.67416,17.32941,2.735946,3.887538,88.73613,5.677310,5.368635,...,0.104417,0.241136,0.238336,0.064262,1.737325,8.279817,0.316324,0.122787,0.829054,1.127895
7,7,20250501_AS_MRC-5_Confluence,14,17.29806,16.88695,3.194657,3.841919,87.74017,5.299525,5.123333,...,0.084954,0.504671,0.146758,0.129890,3.004194,17.425700,0.572962,0.117461,0.659487,0.235604
8,7,20250501_AS_MRC-5_Confluence,16,18.42807,16.36398,3.035607,3.684577,88.85304,5.814235,5.384888,...,0.080583,0.264013,0.000000,0.034555,0.187869,3.115029,0.156933,0.141363,0.627732,0.057638
9,7,20250501_AS_MRC-5_Confluence,18,18.75007,15.80153,4.061611,4.389205,87.94567,5.692607,5.199273,...,0.272277,0.239633,0.169840,0.167381,0.287779,3.914582,0.404147,0.115139,0.765133,0.215731


,Plate_Number,Vessel_Name,Elapsed,"Doxo_P19 LIN12-0C-b2-s2 (A1), Image 1","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 2","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 3","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 4","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 6","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 7","Doxo_P19 LIN12-0C-b2-s2 (A1), Image 8",...,"Blank (C3), Image 8","Blank (C3), Image 9","Blank (C4), Image 1","Blank (C4), Image 2","Blank (C4), Image 3","Blank (C4), Image 4","Blank (C4), Image 6","Blank (C4), Image 7","Blank (C4), Image 8","Blank (C4), Image 9"
0,7,20250501_AS_MRC-5_Confluence,4,15.77353,16.47550,3.036290,3.456963,7.190301,6.018014,9.575639,...,0.129890,0.525090,0.445873,0.272550,0.336607,0.672531,0.466838,0.182610,5.572279,0.658258
1,7,20250501_AS_MRC-5_Confluence,6,16.08801,16.31747,3.001666,3.587808,5.811367,5.851794,10.884910,...,0.238677,0.471140,0.527549,0.139928,0.213478,0.710091,0.336334,0.214161,1.539691,0.213068
2,7,20250501_AS_MRC-5_Confluence,8,16.07449,16.58627,2.832714,3.704177,5.470730,6.220634,10.511090,...,0.238746,0.208493,0.334695,0.227819,0.113500,0.139860,0.396703,0.083042,2.310151,0.615439
3,7,20250501_AS_MRC-5_Confluence,10,17.27983,16.71561,3.338751,3.628578,5.389533,5.053198,9.527699,...,0.511568,0.295769,0.279925,0.027180,0.029092,0.363513,0.357231,0.114183,1.886336,0.309085
4,7,20250501_AS_MRC-5_Confluence,12,16.67416,17.32941,2.735946,3.887538,5.677310,5.368635,11.183900,...,0.242297,0.104417,0.241136,0.238336,0.064262,1.737325,0.316324,0.122787,0.829054,1.127895
5,7,20250501_AS_MRC-5_Confluence,14,17.29806,16.88695,3.194657,3.841919,5.299525,5.123333,11.608190,...,0.121012,0.084954,0.504671,0.146758,0.129890,3.004194,0.572962,0.117461,0.659487,0.235604
6,7,20250501_AS_MRC-5_Confluence,16,18.42807,16.36398,3.035607,3.684577,5.814235,5.384888,10.642410,...,0.161918,0.080583,0.264013,0.000000,0.034555,0.187869,0.156933,0.141363,0.627732,0.057638
7,7,20250501_AS_MRC-5_Confluence,18,18.75007,15.80153,4.061611,4.389205,5.692607,5.199273,13.250790,...,0.264286,0.272277,0.239633,0.169840,0.167381,0.287779,0.404147,0.115139,0.765133,0.215731
8,7,20250501_AS_MRC-5_Confluence,20,17.06348,16.04240,3.649680,4.342425,6.013644,5.123607,11.377980,...,0.237243,0.337563,0.394722,0.182337,0.000000,0.337221,0.632512,0.082974,1.119427,0.416302
9,7,20250501_AS_MRC-5_Confluence,22,18.53515,16.18062,4.364619,3.915401,6.453370,5.496613,12.551490,...,0.289281,0.401415,0.246941,0.139519,0.120739,0.495042,0.466155,0.131324,1.094228,0.448399


,Plate_Number,Vessel_Name,Elapsed,Doxo_P19 LIN12-0C-b2-s2 (A1),Doxo_P19 LIN12-0C-b2-s2 (A2),P34 LIN3-0A-b2-s1-ss2 (A3),P34 LIN3-0A-b2-s1-ss2 (A4),P33 LIN3-0A-b2-s2-ss1-ss1 (B1),P33 LIN3-0A-b2-s2-ss1-ss1 (B2),P29 LIN2-0A-b1-s1-ss1 (B3),...,P24 LIN7-0C-b3-s2 (A3),P24 LIN7-0C-b3-s2 (A4),P21 LIN11-0C-b5 (B1),P21 LIN11-0C-b5 (B2),P20 LIN12-0C-b2-s2 (B3),P20 LIN12-0C-b2-s2 (B4),P13 LIN11-0C-b5-s1 (C1),P13 LIN11-0C-b5-s1 (C2),Blank (C3),Blank (C4)
0,7,20250501_AS_MRC-5_Confluence,4,7.955220,4.453057,3.789882,2.297022,15.315026,12.455320,11.923360,...,9.220321,11.857602,19.490060,19.065059,17.745754,23.016741,19.761686,20.502607,0.245003,1.075943
1,7,20250501_AS_MRC-5_Confluence,6,7.942508,5.295427,3.866812,2.308614,15.446853,12.480333,12.619363,...,9.818227,12.606603,19.600086,19.002469,18.338280,23.410875,20.729056,21.292289,0.351366,0.486787
2,7,20250501_AS_MRC-5_Confluence,8,7.927912,5.101549,3.872924,2.297090,15.604418,12.411394,13.023597,...,10.126535,12.858869,21.086764,19.959202,19.379088,24.950239,22.474090,23.478021,0.251533,0.527651
3,7,20250501_AS_MRC-5_Confluence,10,7.898888,5.580602,4.187635,2.107036,15.596728,12.700902,13.804069,...,10.806731,13.193555,21.837277,20.874440,20.726820,27.152598,25.113431,25.986975,0.335617,0.420818
4,7,20250501_AS_MRC-5_Confluence,12,8.159804,5.535154,3.906660,2.275416,16.368775,13.116711,14.418347,...,11.285246,14.023401,22.959891,22.463445,21.769504,28.608219,27.178124,28.666232,0.188629,0.584640
5,7,20250501_AS_MRC-5_Confluence,14,8.218098,5.115882,3.892096,2.187876,15.969016,12.968051,14.976762,...,11.373010,14.282602,24.273551,23.694983,22.879704,30.507425,28.845570,30.053607,0.170830,0.671378
6,7,20250501_AS_MRC-5_Confluence,16,8.240863,5.426674,3.714608,2.195473,16.351839,12.675518,15.022433,...,12.358405,14.988082,24.736454,24.260920,23.950425,32.434328,30.042099,31.106041,0.133842,0.183763
7,7,20250501_AS_MRC-5_Confluence,18,8.740464,5.445983,3.526432,2.214594,16.711979,13.442389,15.532738,...,12.705367,15.730081,26.955011,25.223369,25.543819,34.046081,32.213528,33.472941,0.258396,0.295598
8,7,20250501_AS_MRC-5_Confluence,20,8.314611,5.462245,3.613392,2.171588,17.060444,13.564631,16.214675,...,13.358648,16.636195,28.572320,26.669569,26.789916,36.324759,34.506068,36.404607,0.219889,0.395687
9,7,20250501_AS_MRC-5_Confluence,22,8.787986,5.440382,3.603251,2.176112,17.207652,14.188292,16.614323,...,13.275342,17.031335,29.657826,28.294430,27.923564,38.532681,38.366561,40.038695,0.244183,0.392793


,Plate_Number,Vessel_Name,Elapsed,Doxo_P19 LIN12-0C-b2-s2,P34 LIN3-0A-b2-s1-ss2,P33 LIN3-0A-b2-s2-ss1-ss1,P29 LIN2-0A-b1-s1-ss1,P25 LIN8-0B-b2B-s1-ss1,P25 LIN8-0B-b2B-s1-ss2,P24 LIN6-0C-b1-s2,P24 LIN7-0C-b3-s2,P21 LIN11-0C-b5,P20 LIN12-0C-b2-s2,P13 LIN11-0C-b5-s1,Blank
0,7,20250501_AS_MRC-5_Confluence,4,6.204138,3.043452,13.885173,12.470451,10.741892,10.111454,11.636856,10.538961,19.277559,20.381248,20.132147,0.660473
1,7,20250501_AS_MRC-5_Confluence,6,6.618967,3.087713,13.963593,12.909930,10.945689,10.148118,12.316467,11.212415,19.301277,20.874578,21.010672,0.419077
2,7,20250501_AS_MRC-5_Confluence,8,6.514730,3.085007,14.007905,13.427965,11.144140,10.400641,12.576178,11.492702,20.522983,22.164663,22.976056,0.389592
3,7,20250501_AS_MRC-5_Confluence,10,6.739745,3.147335,14.148815,14.043284,11.777318,10.549346,13.117491,12.000143,21.355859,23.939709,25.550203,0.378217
4,7,20250501_AS_MRC-5_Confluence,12,6.847479,3.091038,14.742743,14.853613,11.951225,10.904173,13.530128,12.654324,22.711668,25.188861,27.922178,0.386634
5,7,20250501_AS_MRC-5_Confluence,14,6.666990,3.039986,14.468534,15.556482,12.598547,11.347455,14.011252,12.827806,23.984267,26.693564,29.449589,0.421104
6,7,20250501_AS_MRC-5_Confluence,16,6.833769,2.955041,14.513678,15.663114,12.663564,11.592295,14.140299,13.673243,24.498687,28.192376,30.574070,0.158802
7,7,20250501_AS_MRC-5_Confluence,18,7.093223,2.870513,15.077184,16.250998,13.147020,11.803448,14.910346,14.217724,26.089190,29.794950,32.843234,0.276997
8,7,20250501_AS_MRC-5_Confluence,20,6.888428,2.892490,15.312537,16.701262,13.629345,12.192613,15.706223,14.997422,27.620944,31.557337,35.455337,0.307788
9,7,20250501_AS_MRC-5_Confluence,22,7.114184,2.889682,15.697972,17.304277,14.547110,12.503080,16.241288,15.153338,28.976128,33.228122,39.202628,0.318488


Wrote replaced data to: /Users/allielas/HTP_fibroblasts_mitolyso/proliferation_growth_curves/Incucyte_sheets/r7_processed.xlsx


In [ ]:
extra_plate_1 = "r0_2024-01-25"
extra_plate_2 = "r0_2024-03-01"

input_1= f"/Users/allielas/Desktop/Active Projects/Quality control/Incucyte_sheets/{extra_plate_1}.xlsx"
output_1= f"/Users/allielas/Desktop/Active Projects/Quality control/Incucyte_sheets/r{extra_plate_1}_processed.xlsx"
input_2= f"/Users/allielas/Desktop/Active Projects/Quality control/Incucyte_sheets/r{extra_plate_2}.xlsx"
output_2= f"/Users/allielas/Desktop/Active Projects/Quality control/Incucyte_sheets/r{extra_plate_2}_processed.xlsx"
make_average_excel_with_exclusions(input_1,output_1)
make_average_excel_with_exclusions(input_2,output_2)

Input file does not exist: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte_sheets/r0_2024-01-25.xlsx


SystemExit: 2

/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
def load_mappings(path: Union[str, Path]):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Mappings file not found: {path}")
    else:
        # assume CSV/TSV: first two columns are old,new
        df = pd.read_csv(p, dtype=str)
        if df.shape[1] < 2:
            raise ValueError("CSV mappings must have at least two columns (old, new).")
        old_col, new_col = df.columns[0], df.columns[1]
        return {
            str(o): str(n)
            for o, n in zip(df[old_col].fillna(""), df[new_col].fillna(""))
        }

def parse_inline_maps(map_args: List[str]):
    mappings = {}
    for s in map_args:
        if ":" not in s:
            raise ValueError(f"Inline mapping must be OLD:NEW, got: {s!r}")
        old, new = s.split(":", 1)
        mappings[old] = new
    return mappings


def apply_replacements(df: pd.DataFrame, mapping: Dict[str, str]):
    if not mapping:
        return df.copy()
    # For performance: compile mapping items once
    items = list(mapping.items())

    def replace_value(x):
        """
        Docstring for replace_value. Just a simple string replace

        :str x: the string to replace based on the dict
        """
        if isinstance(x, str):
            for old, new in items:
                if old:
                    x = x.replace(old, new)
            return x
        return x

    return df.applymap(replace_value)